# Suite TRN — Translators

Every pair of dialects has its own bidirectional translator (`mbse.Expressions.Translators`), declared as rules
that pair two patterns with holes. Translating co-traverses each rule's pattern with the term to bind the holes, then
instantiates the other side, a writer of the target dialect's term that the caller writes in a transaction. A dialect
with no binding gets a let's value substituted for its name, and imports are dropped where the other side cannot
declare them, or added where the translation needs them (`import numpy as np`). Translations keep an expression's
form; its value follows each dialect's rules, and this suite shows where those agree and where they do not.

In [ ]:
import itertools

import numpy as np

from mbse.Expressions import Framework, Translators
from mbse.Expressions.Dialects import C, Cpp, Excel, Latex, Matlab, Patterns, Python3, SystemVerilog
from mbse.Expressions.Dialects.Patterns import Evaluators as V, Expressions as E
from mbse.Expressions.Dialects.C import Evaluators as CV, Expressions as C, Text as CText
from mbse.Expressions.Dialects.Excel import Evaluators as XV, Expressions as X, Text as XText
from mbse.Expressions.Dialects.Latex import Expressions as L, Text as LText
from mbse.Expressions.Dialects.Matlab import Evaluators as MV, Expressions as M, Text as MText
from mbse.Expressions.Dialects.Python3 import Evaluators as PV, Expressions as P, Text as PText
from mbse.Expressions.Dialects.SystemVerilog import Evaluators as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Translators import Convert, Elementwise, Elide, Hole, Inline, Pairwise, Pattern, Prelude, Rule, holes, \
    renames
from mbse.Expressions.Translators import Patterns_Python3
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()  # every dialect's kinds, and the data the expressions are evaluated over
for module in (Framework, Patterns, Python3, Matlab, Excel, Latex, C, Cpp, SystemVerilog):
    store.transaction(module.install)


def data(txn):
    S = txn.Schemas
    def ints():
        return S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt]))

    return txn.uid("trn.data/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Bus": S.OfObject})
        .spec([
            lambda _: _.Person.name("Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt, "email": S.OfStr}),
            lambda _: _.Bus.name("Bus").ref(True).properties({"pins": ints(), "allowed": ints()}),
        ])]))])


store.transaction(data)


def term(spec):
    """The term value `spec` (a writer of any dialect, a native value as a Patterns literal, or a term) writes."""
    with store.transaction():
        return store.Expressions.Framework.Term(E.DIALECT.spec(spec))


def attributes(spec):
    """The attributes of the term `spec` writes: a literal's `value`, say."""
    return F.form(term(spec)).attributes


PATTERNS, PYTHON, MATLAB, EXCEL, LATEX = E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT
DIALECTS = [PATTERNS, PYTHON, MATLAB, EXCEL, LATEX]
EVALUATE = {PATTERNS: V.OfAny, PYTHON: PV.OfAny, MATLAB: MV.OfAny, EXCEL: XV.OfAny}
NUMPY = Patterns_Python3.NUMPY
person = E.variable("person")
age = person.age
core = term(age.ge(18).and_(age.lt(65).or_(person.has("email").not_())).and_(person.name.ne("x")).or_(
    age.add(1).sub(2).mul(3).neg().gt(0).and_(age.eq(1).not_())).and_(person.has("name").eq(True)).and_(
    E.literal(1.5).le(2.5)))  # every translatable operation and literal type, operands of one type, no let
constraint = term(E.let_("a", age, E.variable("a").ge(18).implies(person.has("email"))))  # a let and implies


def into(source, target, expression):
    """`expression`, a term of `source`, translated to `target` and written: itself when they are the same."""
    return expression if source is target else term(Translators.between(source, target).forward(expression))


with store.transaction():
    people = [store.Person(lambda p: p.name("Ann").age(30).email("a@x")), store.Person(lambda p: p.name("Bob").age(70)),
              store.Person(lambda p: p.name("x").age(10).email("c@x"))]

## TRN-01 · Rules pair patterns; holes bind arguments and attributes

In [ ]:
A, B_, K = *holes("A", "B"), Hole("K", str)
double = Pattern("call", A, A, callable="add")
assert double.size() == 2 and repr(double) == "call(A, A, callable='add')" and double.holes() == {"A": "argument"}
has = Pattern("call", A, Pattern("literal", value=K), callable="has")
assert has.size() == 3 and has.holes() == {"A": "argument", "K": "attribute"}
with raises(ValueError, match="call(A, B, callable='f') needs the argument hole B, which call(A, callable='f') does not bind"):
    Rule(Pattern("call", A, callable="f"), Pattern("call", A, B_, callable="f"))
with raises(ValueError, match="variable(name=A) needs the attribute hole A"):
    Rule(Pattern("call", A, callable="f"), Pattern("variable", name=A))
Rule(Pattern("call", A, callable="f"), Pattern("call", A, B_, callable="f"), "backward")  # only the other way needs B
with raises(ValueError, match="direction must be 'both', 'forward' or 'backward', got 'up'"):
    Rule(double, double, "up")
with raises(ValueError, match="direction must be 'both', 'forward' or 'backward', got 'up'"):
    Elementwise("list", "list", direction="up")
for declare in [lambda: Inline("let", "up"), lambda: Elide("import", "up"), lambda: Prelude(Pattern("import", A), "up")]:
    with raises(ValueError, match="side must be 'left' or 'right', got 'up'"):
        declare()
for pattern in [Pattern("import", module="numpy"), Pattern("import", Pattern("name", name="x")),
                Pattern("import", A, module=Hole("M"))]:
    with raises(ValueError, match="a prelude is an import whose one argument is a hole"):
        Prelude(pattern, "right")
assert [(r.left.attributes, r.right.attributes) for r in renames("call", "callable", "compare", "operator",
                                                                  {"eq": "=="}, 2)] == [
    ({"callable": "eq"}, {"operator": "=="})]
# A conversion translates literals by functions of their attributes, which give None where there is no counterpart.
double_ints = Convert("literal", "literal", lambda a: {"value": a["value"] * 2} if type(a["value"]) is int else None,
                      lambda a: {"value": a["value"] // 2})
doubling = Pairwise(PATTERNS, PATTERNS, [double_ints])
assert attributes(doubling.forward(term(3)))["value"] == 6 and attributes(doubling.backward(term(6)))["value"] == 3
assert attributes(doubling.inverse().forward(term(6)))["value"] == 3
assert attributes(doubling.inverse().backward(term(3)))["value"] == 6
with raises(ValueError, match="Patterns literal 'a' has no Patterns counterpart"):
    doubling.forward(term("a"))
for left, right, named_ in [("call", "literal", "Patterns 'call'"), ("literal", "nothing", "Patterns 'nothing'")]:
    with raises(ValueError, match=f"a conversion translates literals, and {named_} is not one"):
        Pairwise(PATTERNS, PATTERNS, [Convert(left, right, double_ints.forward, double_ints.backward)])

# A translator within one dialect is a rewriting: holes bound twice must bind equal terms, or equal values.
simplify = Pairwise(PATTERNS, PATTERNS, [
    Rule(double, Pattern("call", A, Pattern("literal", value=2), callable="mul"), "forward"),
    Rule(Pattern("call", Pattern("literal", value=K), Pattern("literal", value=K), callable="eq"), Pattern("literal", value=True), "forward"),
    Rule(Pattern("call", A, B_, callable="add"), Pattern("call", A, B_, callable="add"), "forward"),
    Rule(Pattern("call", Pattern("literal", value=Hole("V")), Pattern("literal", value=Hole("W")), callable="eq"),
         Pattern("literal", value=False), "forward"),
    Rule(Pattern("variable", name=Hole("N")), Pattern("variable", name=Hole("N"))),
    Rule(Pattern("literal", value=Hole("V")), Pattern("literal", value=Hole("V")), "forward"),
])
x, y = E.variable("x"), E.variable("y")
assert F.same(term(simplify.forward(term(x.add(x)))), term(x.mul(2)))
assert F.same(term(simplify.forward(term(x.add(y)))), term(x.add(y)))
assert attributes(simplify.forward(term(E.literal("a").eq("a"))))["value"] is True
assert attributes(simplify.forward(term(E.literal("a").eq("b"))))["value"] is False
assert attributes(simplify.forward(term(E.literal(1).eq(1))))["value"] is False  # K holds only a str
assert repr(simplify) == "<translator Patterns <-> Patterns>"
# A prelude of an import that binds no names never wraps a translation.
plain = Pairwise(PATTERNS, MATLAB, [Rule(Pattern("variable", name=Hole("N")), Pattern("variable", name=Hole("N"))),
                                 Prelude(Pattern("import", A, name="geo.*"), "right")])
assert F.same(term(plain.forward(term(x))), term(M.identifier("x")))  # a Matlab term
# A translation is a writer: written where its caller wants it, as an expression's root.
doubled = simplify.forward(term(x.add(x)))
with store.transaction():
    expression = store.Expressions.Framework.Expression(lambda e: e.root(doubled))
assert F.same(F.term(expression), term(x.mul(2))) and isinstance(simplify.forward(term(x)), F.Writer)

## TRN-02 · Every pair translates both ways; forms round-trip where both dialects have them

In [ ]:
assert len(Translators.TRANSLATORS) == 13  # every pair of the five, and Patterns with C, Cpp and SystemVerilog
for source, target in itertools.permutations(DIALECTS, 2):
    translator = Translators.between(source, target)
    assert translator.left() is source and translator.right() is target
    start = into(PATTERNS, source, core)
    there = term(translator.forward(start))
    assert target.validate(there, bound={"person"}, core=True) == [], (source.name(), target.name())
    assert F.same(term(translator.backward(there)), start), (source.name(), target.name())  # round trip
    assert F.same(term(translator.inverse().backward(start)), there)
with raises(LookupError, match="no translator between Patterns and Patterns"):
    Translators.between(PATTERNS, PATTERNS)

assert PText.ToText(into(PATTERNS, PYTHON, constraint)) == "(lambda a: hasattr(person, 'email') if a >= 18 else True)(person.age)"
assert MText.ToText(into(PATTERNS, MATLAB, constraint)) == '~(person.age >= 18) || isfield(person, "email")'
assert XText.ToText(into(PATTERNS, EXCEL, constraint)) == "=LET(a, person.age, IF(a >= 18, NOT(ISERROR(person.email)), TRUE))"
assert LText.ToText(into(PATTERNS, LATEX, constraint)) == (
    "a \\geq 18 \\implies \\operatorname{has}(\\mathit{person}, \\text{email}) \\quad \\text{where } a = \\mathit{person}.\\mathit{age}")
assert LText.ToText(into(PYTHON, LATEX, term(P.binop("/", P.name("x"), 2)))) == "\\frac{x}{2}"
# Python, Excel and Latex have lets and implication, so the constraint round-trips; Matlab inlines the let and writes implies
# with ||.
for other in (PYTHON, EXCEL, LATEX):
    assert F.same(into(other, PATTERNS, into(PATTERNS, other, constraint)), constraint)
back = into(MATLAB, PATTERNS, into(PATTERNS, MATLAB, constraint))
assert F.same(back, term(age.ge(18).not_().or_(person.has("email"))))


# NumPy style is a second translator between Patterns and Python: numpy's functions, over columns, importing numpy.
styled = term(NUMPY.forward(constraint))
assert PText.ToText(styled) == (
    "import numpy as np\n(lambda a: np.where(np.greater_equal(a, 18), "
    "np.logical_not(np.ma.getmaskarray(person['email'])), True))(person['age'])")
assert F.same(term(NUMPY.backward(styled)), constraint) and F.same(term(NUMPY.backward(term(NUMPY.forward(core)))), core)
assert PText.ToText(term(NUMPY.forward(term(E.literal(1).add(2))))) == "import numpy as np\nnp.add(1, 2)"
assert PText.ToText(term(NUMPY.forward(term(1)))) == "1" and PText.ToText(term(NUMPY.forward(term(person)))) == "person"  # no np, no import
assert F.same(term(NUMPY.inverse().forward(styled)), constraint)
assert attributes(NUMPY.inverse().backward(term(1)))["value"] == 1
imported = term(P.import_("math", P.importfrom("numpy", "pi", P.compare(">=", P.attribute(P.name("person"), "age"), 18))))
# imports the other side cannot declare are dropped
for target in (PATTERNS, MATLAB, EXCEL):
    assert F.same(into(PYTHON, target, imported), into(PATTERNS, target, term(age.ge(18))))
assert XText.ToText(into(MATLAB, EXCEL, term(M.import_("pkg.*", M.binary(">", 1, 0))))) == "=1 > 0"
assert F.same(into(MATLAB, PATTERNS, term(M.import_("pkg.*", M.constant(1)))), term(1))
with raises(ValueError, match="an import needs a body"):
    Translators.between(PYTHON, MATLAB).forward(term(P.DIALECT.spec(lambda t: getattr(t, "import")(lambda i: i.module("math")))))
with raises(ValueError, match="Python3 call has no Patterns counterpart"):
    Translators.between(PYTHON, PATTERNS).forward(term(P.import_("numpy", P.call("np.add", 1, 2), alias="np")))  # read by NUMPY

## TRN-03 · Translating the same pair of dialects by any route gives the same expression

In [ ]:
for a, b, c in itertools.permutations(DIALECTS, 3):
    start = into(PATTERNS, a, core)
    direct = into(a, c, start)
    routed = into(b, c, into(a, b, start))
    assert F.same(direct, routed), (a.name(), b.name(), c.name())

## TRN-04 · Co-traversal translates each term once, keeps sharing, and traces the correspondence

In [ ]:
# A term is a nested value, so what was shared is written at each use: each use is translated once, and the trace pairs
# each source term with the writer of its translation, a term after the terms within it.
shared = term(E.operation("add", age, age))
trace = []
out = term(Translators.between(PATTERNS, EXCEL).forward(shared, trace))
left, right = F.form(out).arguments
assert F.same(left, right) and XText.ToText(out) == "=person.age + person.age"
assert [(F.kind_of(s)[0], F.kind_of(term(t))[0]) for s, t in trace] == [
    ("variable", "variable"), ("call", "field"), ("variable", "variable"), ("call", "field"),
    ("call", "infix")]
assert F.same(trace[-1][0], shared) and F.same(term(trace[-1][1]), out) and F.same(term(trace[1][1]), left)
a = E.variable("a")
inlined = term(Translators.between(PATTERNS, MATLAB).forward(term(E.let_("a", age, a.add(a))), trace := []))
assert MText.ToText(inlined) == "person.age + person.age" and F.same(*F.form(inlined).arguments)
assert [F.kind_of(s)[0] for s, _ in trace] == ["variable", "call", "variable", "variable", "call", "let"]  # the value once
shadowed = term(E.let_("a", 1, E.variable("a").add(E.let_("a", 2, E.variable("a")))))  # the innermost binding wins
assert MText.ToText(into(PATTERNS, MATLAB, shadowed)) == "1 + 2"
assert MV.OfAny(into(PATTERNS, MATLAB, shadowed)) == V.OfAny(shadowed) == 3
assert MText.ToText(into(EXCEL, MATLAB, term(X.let_("n", 2, X.infix("*", X.name("n"), X.name("n")))))) == "2 .* 2"
with raises(TypeError, match="not a term: 7"):
    Translators.between(PATTERNS, MATLAB).forward(7)
with raises(ValueError, match="a let needs a value"):
    Translators.between(PATTERNS, MATLAB).forward(term(lambda t: t.let(lambda l: l.name("a").body(E.variable("a")))))
with raises(ValueError, match="a let needs a body"):
    Translators.between(EXCEL, MATLAB).forward(term(X.DIALECT.spec(lambda t: t.let(lambda l: l.name("a").value(X.constant(1))))))

## TRN-05 · What a dialect cannot say is refused, naming the term

In [ ]:
for translate, expression, message in [
    (Translators.between(PATTERNS, EXCEL).forward, E.literal(b"\x00"), "Patterns literal b'\\x00' has no Excel counterpart"),
    (Translators.between(PATTERNS, PYTHON).forward, lambda t: t.literal(lambda l: l), "Patterns literal None has no Python3 counterpart"),
    (Translators.between(PATTERNS, MATLAB).forward, E.operation("entries", person, "ports"), "Patterns call 'entries' has no Matlab counterpart"),
    (Translators.between(PATTERNS, PYTHON).forward, person.get(E.variable("key")), "Patterns call 'get' has no Python3 counterpart"),
    (NUMPY.forward, E.operation("has", person, 7), "Patterns call 'has'"),
    (Translators.between(PATTERNS, PYTHON).forward, E.operation("eq", 1, 2, 3), "Patterns call 'eq' has no Python3 counterpart"),
    (Translators.between(EXCEL, PATTERNS).forward, X.function("ISERROR", 1), "Excel call 'ISERROR' has no Patterns counterpart"),
    (Translators.between(EXCEL, PATTERNS).forward, X.function("IF", True, 1, 2), "Excel call 'IF' has no Patterns counterpart"),
    (Translators.between(EXCEL, PATTERNS).forward, X.DIALECT.spec(lambda t: t.let(lambda l: l.name("a").value(X.constant(1)))), "Excel let has no Patterns counterpart"),
    (Translators.between(PYTHON, MATLAB).forward, P.ifexp(True, 1, 2), "Python3 ifexp has no Matlab counterpart"),
    (Translators.between(EXCEL, MATLAB).forward, X.cell("A1"), "Excel cell has no Matlab counterpart"),
    (Translators.between(PATTERNS, LATEX).forward, E.literal(b"\x00"), "Patterns literal b'\\x00' has no Latex counterpart"),
    (Translators.between(LATEX, PATTERNS).forward, L.frac(1, 2), "Latex frac has no Patterns counterpart"),
    (Translators.between(PYTHON, LATEX).forward, P.ifexp(True, 1, 2), "Python3 ifexp has no Latex counterpart"),
    (Translators.between(MATLAB, PATTERNS).forward, M.call("isfield", M.identifier("s"), M.identifier("k")), "Matlab call 'isfield'"),
    (Translators.between(PYTHON, PATTERNS).forward, P.DIALECT.spec(lambda t: t.call(lambda c: c.arguments([P.name("x"), P.constant("k")]))),
     "Python3 call has no Patterns counterpart"),  # a call of no function: not hasattr's
    (Translators.between(PYTHON, MATLAB).forward, P.constant(b"\x00"), "Python3 literal b'\\x00' has no Matlab counterpart"),
]:
    with raises(ValueError, match=message):
        translate(term(expression))
# A literal translates by its value: its spelling is its own dialect's, and one with no value has none to translate.
spelled = term(P.DIALECT.spec(lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(16)).text("0x10"))))
assert PText.ToText(spelled) == "0x10" and MText.ToText(into(PYTHON, MATLAB, spelled)) == "16"
assert F.form(into(PYTHON, PATTERNS, spelled)) == F.Form("literal", {"value": 16})
with raises(ValueError, match="Python3 literal None has no Matlab counterpart"):
    Translators.between(PYTHON, MATLAB).forward(term(P.DIALECT.spec(lambda t: t.literal(lambda r: r.text("08").problems(["bad"])))))

## TRN-06 · Translations evaluate alike where the dialects' rules agree, and apart where they do not

In [ ]:
EVALUATED = [d for d in DIALECTS if d in EVALUATE]  # Latex has no evaluator
translations = {d: into(PATTERNS, d, core) for d in EVALUATED}
lets = {d: into(PATTERNS, d, constraint) for d in EVALUATED}
for someone in people:
    for expressions in (translations, lets):
        values = {d.name(): bool(EVALUATE[d](e, {"person": someone})) for d, e in expressions.items()}
        assert len(set(values.values())) == 1, (someone.name.get(), values)
columns = {"age": np.array([30, 70, 10]), "name": np.array(["Ann", "Bob", "x"]),
           "email": np.ma.array(["a@x", "", "c@x"], mask=[False, True, False])}
with_numpy = PV.Scope({"person": columns}, modules={"numpy": np})
assert PV.OfAny(term(NUMPY.forward(core)), with_numpy).tolist() == [
    bool(V.OfAny(core, {"person": someone})) for someone in people]  # numpy evaluates all three at once

with store.transaction():
    nobody = store.Person(lambda p: p.name("Nobody"))  # no age: each dialect's own notion of a missing value
missing = term(age.ge(18))
assert V.OfAny(missing, {"person": nobody}) is None  # Patterns: unknown
absent = PV.Scope({"person": {"age": np.ma.array([0], mask=[True])}}, modules={"numpy": np})
assert PV.OfAny(term(NUMPY.forward(missing)), absent).mask.tolist() == [True]  # NumPy: masked
with raises(AttributeError, match="'Person' object has no attribute 'age'"):  # Python raises
    PV.OfAny(into(PATTERNS, PYTHON, missing), {"person": nobody})
assert XV.OfAny(into(PATTERNS, EXCEL, missing), {"person": nobody}) == XV.Error("#FIELD!")  # an error value
with raises(KeyError, match='Unrecognized field name "age".'):  # MATLAB raises
    MV.OfAny(into(PATTERNS, MATLAB, missing), {"person": nobody})
mixed, cased = term(E.literal(1).eq(1.0)), term(E.literal("a").eq("A"))
assert [V.OfAny(mixed), V.OfAny(cased)] == [None, False]  # Patterns never coerces, and compares text exactly
assert [bool(EVALUATE[d](into(PATTERNS, d, mixed))) for d in EVALUATED[1:]] == [True, True, True]
assert [bool(EVALUATE[d](into(PATTERNS, d, cased))) for d in EVALUATED[1:]] == [False, False, True]

## TRN-07 · The bitwise operations translate between Patterns, Python, MATLAB and Excel, and evaluate alike on non-negative integers

In [ ]:
x = E.variable("x")
bits = term(x.bitand(12).bitor(x.shr(2)).bitxor(x.shl(1)))
RENDER = {PYTHON: PText.ToText, MATLAB: MText.ToText, EXCEL: XText.ToText}
expected = {PYTHON: "(x & 12 | x >> 2) ^ x << 1", MATLAB: "bitxor(bitor(bitand(x, 12), bitshift(x, -2)), bitshift(x, 1))",
            EXCEL: "=BITXOR(BITOR(BITAND(x, 12), BITRSHIFT(x, 2)), BITLSHIFT(x, 1))"}
for dialect, source in expected.items():
    translator = Translators.between(PATTERNS, dialect)
    translated = term(translator.forward(bits))
    assert RENDER[dialect](translated) == source and F.same(term(translator.inverse().forward(translated)), bits)
    assert EVALUATE[dialect](translated, {"x": 13}) == V.OfAny(bits, {"x": 13}) == 21
# Between the other dialects too, by any route.
for a, b in [(PYTHON, MATLAB), (PYTHON, EXCEL), (MATLAB, EXCEL)]:
    source = into(PATTERNS, a, bits)
    target = into(a, b, source)
    assert F.same(target, into(PATTERNS, b, bits)) and F.same(into(b, a, target), source)
# bitnot is Python's ~ and NumPy's invert; MATLAB and Excel have none for their numbers.
assert PText.ToText(into(PATTERNS, PYTHON, term(x.bitnot()))) == "~x"
assert PText.ToText(term(NUMPY.forward(term(x.bitnot().shl(1))))) == "import numpy as np\nnp.left_shift(np.invert(x), 1)"
with raises(ValueError, match="Patterns call 'bitnot' has no Matlab counterpart"):
    Translators.between(PATTERNS, MATLAB).forward(term(x.bitnot()))
with raises(ValueError, match="Patterns call 'bitnot' has no Excel counterpart"):
    Translators.between(PATTERNS, EXCEL).forward(term(x.bitnot()))
with raises(ValueError, match="Python3 unaryop '~' has no Matlab counterpart"):
    Translators.between(PYTHON, MATLAB).forward(term(PText.FromText("~x")))
# Patterns' and Python3's integers are unbounded and signed; MATLAB's and Excel's bit functions are not.
mask = term(x.bitand(1))
assert V.OfAny(mask, {"x": -1}) == 1 and PV.OfAny(into(PATTERNS, PYTHON, mask), {"x": -1}) == 1
assert XV.OfAny(into(PATTERNS, EXCEL, mask), {"x": -1}) == XV.Error("#NUM!")

## TRN-08 · Patterns and C, and Patterns and C++, translate into each other by one rule set; lets are inlined, and `has`, `->` and typed constants have no counterpart

In [ ]:
CCPP, CPP = C.DIALECT, Cpp.Expressions.DIALECT
x, person = E.variable("x"), E.variable("person")
constraint_c = term(E.let_("a", person.age, E.variable("a").ge(18).and_(person.width.shl(2).bitand(255).ne(0)).implies(person.name.eq("x"))))
translator = Translators.between(PATTERNS, CCPP)
translated = term(translator.forward(constraint_c))
assert CText.ToText(translated) == '!(person.age >= 18 && (person.width << 2 & 255) != 0) || person.name == "x"'
expected = term(E.operation("or", E.operation("not", person.age.ge(18).and_(person.width.shl(2).bitand(255).ne(0))), person.name.eq("x")))
assert F.same(term(translator.inverse().forward(translated)), expected)
every = term(x.add(1).sub(2).mul(3).neg().bitor(x.bitxor(1)).bitand(x.bitnot()).shr(1).lt(x).or_(x.le(1).and_(x.gt(2).not_())).ne(
    x.ge(0).eq(E.literal(True))))
assert F.same(into(CCPP, PATTERNS, into(PATTERNS, CCPP, every)), every) and CText.ToText(into(PATTERNS, CCPP, term(1.5))) == "1.5"
check = term(x.add(1).mul(2).gt(5).and_(x.lt(9)))
for value in [3, 9]:  # C's ints and Patterns' agree where neither overflows
    assert CV.OfAny(into(PATTERNS, CCPP, check), {"x": value}) == V.OfAny(check, {"x": value})
# A typed constant has no Patterns counterpart yet: a term carries no domain, and its conversions wait for domain inference.
for constant, plain in [(C.constant(True, "bool"), True), (C.constant(1.5), 1.5), (C.constant(5), 5)]:  # untyped
    assert F.same(into(CCPP, PATTERNS, term(constant)), term(plain))
for expression, source, message in [
    (person.has("email"), PATTERNS, "Patterns call 'has' has no C counterpart"),
    (E.literal(b"\x01"), PATTERNS, "Patterns literal b'\\x01' has no C counterpart"),
    (E.list_(1), PATTERNS, "Patterns list has no C counterpart"),
    (C.constant(5, "unsigned int"), CCPP, "C constant 5 of unsigned int has no Patterns counterpart"),
    (C.constant(1.5, "float"), CCPP, "C constant 1.5 of float has no Patterns counterpart"),
    (C.constant(5, "word"), CCPP, "C constant 5 has no Patterns counterpart"),
    (C.member(C.identifier("x"), "a", "->"), CCPP, "C member has no Patterns counterpart"),
]:
    with raises(ValueError, match=message):
        Translators.between(source, PATTERNS if source is CCPP else CCPP).forward(term(expression))
# C++: the same rules (Patterns_C.RULES), over the kinds it shares with C, writing C++'s terms
assert Translators.Patterns_Cpp.RULES is Translators.Patterns_C.RULES  # one rule set
in_cpp = into(PATTERNS, CPP, constraint_c)
assert F.dialect_of(in_cpp) == CPP.union and Cpp.Text.ToText(in_cpp) == CText.ToText(translated)
assert F.same(into(CPP, PATTERNS, in_cpp), expected) and F.same(into(CPP, PATTERNS, into(PATTERNS, CPP, every)), every)
assert Cpp.Evaluators.OfAny(into(PATTERNS, CPP, check), {"x": 3}) == V.OfAny(check, {"x": 3})
with raises(ValueError, match="Cpp constant 5 of unsigned int has no Patterns counterpart"):
    Translators.between(CPP, PATTERNS).forward(term(Cpp.Expressions.constant(5, "unsigned int")))
with raises(LookupError, match="no translator between C and Cpp"):  # C++ is not a superset of C
    Translators.between(CCPP, CPP)

## TRN-09 · Patterns and SystemVerilog translate into each other; lets are inlined, bools are single bits, and `has`, sized vectors and 4-state comparisons have no counterpart

In [ ]:
SYSTEMVERILOG = SV.DIALECT
x, person = E.variable("x"), E.variable("person")
constraint_sv = term(E.let_("a", person.age, E.variable("a").ge(18).and_(person.width.shr(2).bitand(255).ne(0)).implies(person.name.eq("x"))))
translator = Translators.between(PATTERNS, SYSTEMVERILOG)
translated = term(translator.forward(constraint_sv))
assert SVText.ToText(translated) == 'person.age >= 18 && (person.width >>> 2 & 255) != 0 -> person.name == "x"'
expected = term(person.age.ge(18).and_(person.width.shr(2).bitand(255).ne(0)).implies(person.name.eq("x")))
assert F.same(term(translator.inverse().forward(translated)), expected)
every = term(x.add(1).sub(2).mul(3).neg().bitor(x.bitxor(1)).bitand(x.bitnot()).shl(1).lt(x).or_(x.le(1).and_(x.gt(2).not_())).ne(
    x.ge(0).eq(E.literal(True)).or_(E.literal(False))))
assert F.same(into(SYSTEMVERILOG, PATTERNS, into(PATTERNS, SYSTEMVERILOG, every)), every)
assert SVText.ToText(into(PATTERNS, SYSTEMVERILOG, term(E.literal(True).and_(E.literal(1.5).lt(2))))) == "1'b1 && 1.5 < 2"
check = term(x.add(1).mul(2).gt(5).and_(x.lt(9)))
for value in [3, 9]:  # SystemVerilog's integers and Patterns' agree where neither overflows
    assert SVV.OfAny(into(PATTERNS, SYSTEMVERILOG, check), {"x": value}).integer() == int(V.OfAny(check, {"x": value}))
# A sized vector has no Patterns counterpart yet but a bool's single bit: its conversions wait for domain inference.
assert F.same(into(SYSTEMVERILOG, PATTERNS, term(SV.vector("1"))), term(True)) and F.same(into(SYSTEMVERILOG, PATTERNS, term(SV.vector("0"))), term(False))
for expression, source, message in [
    (person.has("email"), PATTERNS, "Patterns call 'has' has no SystemVerilog counterpart"),
    (E.mapping(("a", 1)), PATTERNS, "Patterns mapping has no SystemVerilog counterpart"),
    (SV.vector("11001000", base="d"), SYSTEMVERILOG, "SystemVerilog vector '11001000' of logic [7:0] has no Patterns counterpart"),
    (SV.vector("10xz"), SYSTEMVERILOG, "SystemVerilog vector '10xz' of logic [3:0] has no Patterns counterpart"),
    (SV.binary("===", SV.identifier("x"), 1), SYSTEMVERILOG, "SystemVerilog binary '===' has no Patterns counterpart"),
]:
    with raises(ValueError, match=message):
        Translators.between(source, PATTERNS if source is SYSTEMVERILOG else SYSTEMVERILOG).forward(term(expression))

## TRN-10 · Collections translate to Python's builtins and generator expressions, and evaluate alike over lists

In [ ]:
PY = Translators.between(PATTERNS, PYTHON)
bus, p = E.variable("bus"), E.variable("p")
collections = term(bus.pins.all("p", p.gt(0)).and_(bus.pins.any("p", p.in_(bus.allowed))).and_(
    bus.pins.count_where("p", p.ge(2)).le(bus.pins.count())).and_(
    bus.pins.item(0).add(bus.pins.sum()).lt(bus.pins.max().mul(bus.pins.min()).add(100))))
there = term(PY.forward(collections))
assert PText.ToText(there) == ("all(p > 0 for p in bus.pins) and any(p in bus.allowed for p in bus.pins) and "
                               "sum(1 for p in bus.pins if p >= 2) <= len(bus.pins) and "
                               "bus.pins[0] + sum(bus.pins) < max(bus.pins) * min(bus.pins) + 100")
assert F.same(term(PY.backward(there)), collections) and F.same(term(PY.inverse().forward(there)), collections)
unique = term(bus.pins.unique())  # written as an idiom, which does not read back
assert PText.ToText(term(PY.forward(unique))) == "len(set(bus.pins)) == len(bus.pins)"
buses = []
with store.transaction():
    for pins, allowed in [([1, 2, 3], [3]), ([1, 1], [5]), ([0, 2], [2])]:
        buses.append(store.Bus(lambda b, pins=pins, allowed=allowed: b.pins(pins).allowed(allowed)))
for b in buses:
    for expression in (collections, unique):
        assert V.OfAny(expression, {"bus": b}) == PV.OfAny(term(PY.forward(expression)), {"bus": b}), b
# The container constructors are Python's displays: a list display and a dict display, evaluating alike.
built = term(E.list_(bus.pins.count(), 2).item(0).add(E.mapping(("a", 1), ("b", E.list_(3))).item("b").count()))
shown = term(PY.forward(built))
assert PText.ToText(shown) == "[len(bus.pins), 2][0] + len({'a': 1, 'b': [3]}['b'])"
assert F.same(term(PY.backward(shown)), built) and PText.ToText(term(PText.FromText(PText.ToText(shown)))) == PText.ToText(shown)
assert all(PV.OfAny(shown, {"bus": b}) == V.OfAny(built, {"bus": b}) for b in buses)
for expression, translator, message in [
    (bus.entries("ports"), PY, "Patterns call 'entries' has no Python3 counterpart"),
    (bus.pins.count_where("p", p.gt(0)), NUMPY, "Patterns quantifier 'count' has no Python3 counterpart"),
    (P.call("all", P.name("xs")), PY.inverse(), "Python3 call has no Patterns counterpart"),
    (P.call("sum", P.generator("p", P.name("xs"), P.name("p"))), PY.inverse(), "Python3 generator has no Patterns counterpart"),
    (PY.forward(unique), PY.inverse(), "Python3 call has no Patterns counterpart"),
]:
    with raises(ValueError, match=message):
        translator.forward(term(expression))

## TRN-11 · Collections translate to SystemVerilog's array methods, and evaluate alike over arrays

In [ ]:
SVT = Translators.between(PATTERNS, SV.DIALECT)
there = term(SVT.forward(collections))  # TRN-10's
assert SVText.ToText(there) == ("bus.pins.and(p) with (p > 0) && bus.pins.or(p) with (p inside {bus.allowed}) && "
                                "bus.pins.sum(p) with (int'(p >= 2)) <= bus.pins.size() && "
                                "bus.pins[0] + bus.pins.sum() < bus.pins.max()[0] * bus.pins.min()[0] + 100")
assert F.same(term(SVT.backward(there)), collections)
assert SVText.ToText(term(SVT.forward(unique))) == "bus.pins.unique().size() == bus.pins.size()"  # an idiom, which does not read back
for b in buses:
    for expression in (collections, unique):
        assert SVV.OfAny(term(SVT.forward(expression)), {"bus": b}).integer() == int(V.OfAny(expression, {"bus": b})), b
xs = SV.identifier("xs")
for expression, translator, message in [
    (bus.entries("ports"), SVT, "Patterns call 'entries' has no SystemVerilog counterpart"),
    (SV.method(xs, "product"), SVT.inverse(), "SystemVerilog method 'product' has no Patterns counterpart"),
    (SV.iterate(xs, "xor", "p", SV.identifier("p")), SVT.inverse(), "SystemVerilog iterate 'xor' has no Patterns counterpart"),
    (SVT.forward(unique), SVT.inverse(), "SystemVerilog method 'unique' has no Patterns counterpart"),
]:
    with raises(ValueError, match=message):
        translator.forward(term(expression))

## TRN-12 · Collections translate to MATLAB's arrays: arrayfun, indexing from 1 and array functions, and evaluate alike

In [ ]:
MT = Translators.between(PATTERNS, MATLAB)
there = term(MT.forward(collections))  # TRN-10's
assert MText.ToText(there) == ("all(arrayfun(@(p) p > 0, bus.pins)) && any(arrayfun(@(p) ismember(p, bus.allowed), bus.pins)) && "
                               "nnz(arrayfun(@(p) p >= 2, bus.pins)) <= numel(bus.pins) && "
                               "bus.pins(0 + 1) + sum(bus.pins) < max(bus.pins) .* min(bus.pins) + 100")
assert F.same(term(MT.backward(there)), collections)
assert MText.ToText(term(MT.forward(unique))) == "numel(unique(bus.pins)) == numel(bus.pins)"  # an idiom, which does not read back
for b in buses:
    for expression in (collections, unique):
        assert MV.OfAny(term(MT.forward(expression)), {"bus": b}) == V.OfAny(expression, {"bus": b}), b
# A list is a concatenation, `[a, b]`, evaluating alike; a mapping has no counterpart.
listed = term(E.list_(bus.pins.count(), 2).sum())
assert MText.ToText(term(MT.forward(listed))) == "sum([numel(bus.pins), 2])" and F.same(term(MT.backward(term(MT.forward(listed)))), listed)
assert all(MV.OfAny(term(MT.forward(listed)), {"bus": b}) == V.OfAny(listed, {"bus": b}) for b in buses)
with raises(ValueError, match="Patterns mapping has no Matlab counterpart"):
    MT.forward(term(E.mapping(("a", 1))))
xs = M.identifier("xs")
for expression, translator, message in [
    (bus.entries("ports"), MT, "Patterns call 'entries' has no Matlab counterpart"),
    (M.index(xs, 1), MT.inverse(), "Matlab index has no Patterns counterpart"),  # only xs(i + 1) reads back
    (M.call("all", xs), MT.inverse(), "Matlab call 'all' has no Patterns counterpart"),
    (MT.forward(unique), MT.inverse(), "Matlab call 'unique' has no Patterns counterpart"),
]:
    with raises(ValueError, match=message):
        translator.forward(term(expression))

## TRN-13 · Collections translate to Excel's arrays: MAP and LAMBDA, INDEX and MATCH and array functions, and evaluate alike

In [ ]:
XT = Translators.between(PATTERNS, EXCEL)
there = term(XT.forward(collections))  # TRN-10's
assert XText.ToText(there) == (
    "=AND(AND(AND(AND(MAP(bus.pins, LAMBDA(p, p > 0))), OR(MAP(bus.pins, LAMBDA(p, ISNUMBER(MATCH(p, bus.allowed, 0)))))), "
    "SUM(MAP(bus.pins, LAMBDA(p, IF(p >= 2, 1, 0)))) <= ROWS(bus.pins)), "
    "INDEX(bus.pins, 0 + 1) + SUM(bus.pins) < MAX(bus.pins) * MIN(bus.pins) + 100)")
assert F.same(term(XT.backward(there)), collections)
assert XText.ToText(term(XT.forward(unique))) == "=ROWS(UNIQUE(bus.pins)) = ROWS(bus.pins)"  # an idiom, which does not read back
for b in buses:
    for expression in (collections, unique):
        assert XV.OfAny(term(XT.forward(expression)), {"bus": b}) == V.OfAny(expression, {"bus": b}), b
# A list is `VSTACK(a, b)`, a column of its elements, evaluating alike; a mapping has no counterpart.
listed = term(E.list_(bus.pins.count(), 2).sum())
assert XText.ToText(term(XT.forward(listed))) == "=SUM(VSTACK(ROWS(bus.pins), 2))" and F.same(term(XT.backward(term(XT.forward(listed)))), listed)
assert all(XV.OfAny(term(XT.forward(listed)), {"bus": b}) == V.OfAny(listed, {"bus": b}) for b in buses)
with raises(ValueError, match="Patterns mapping has no Excel counterpart"):
    XT.forward(term(E.mapping(("a", 1))))
xs = X.name("xs")
for expression, translator, message in [
    (bus.entries("ports"), XT, "Patterns call 'entries' has no Excel counterpart"),
    (X.function("INDEX", xs, 1), XT.inverse(), "Excel call 'INDEX' has no Patterns counterpart"),  # only INDEX(xs, i + 1) reads back
    (X.function("AND", xs), XT.inverse(), "Excel call 'AND' has no Patterns counterpart"),
    (XT.forward(unique), XT.inverse(), "Excel call 'UNIQUE' has no Patterns counterpart"),
]:
    with raises(ValueError, match=message):
        translator.forward(term(expression))

## TRN-14 · Collections translate directly between Python, MATLAB and Excel, as they do through Patterns

In [ ]:
for left, right in [(PYTHON, MATLAB), (PYTHON, EXCEL), (MATLAB, EXCEL)]:
    source = into(PATTERNS, left, collections)  # TRN-10's
    there = into(left, right, source)
    assert F.same(there, into(PATTERNS, right, collections)), (left.name(), right.name())
    assert F.same(into(right, left, there), source), (left.name(), right.name())
with raises(ValueError, match="Python3 call has no Matlab counterpart"):  # unique's idiom does not translate
    Translators.between(PYTHON, MATLAB).forward(into(PATTERNS, PYTHON, unique))
listed = term(E.list_(1, E.list_(2, 3).count()).count())  # a list: a list display, a concatenation, VSTACK
for left, right in [(PYTHON, MATLAB), (PYTHON, EXCEL), (MATLAB, EXCEL)]:
    source = into(PATTERNS, left, listed)
    assert F.same(into(left, right, source), into(PATTERNS, right, listed)) and F.same(into(right, left, into(left, right, source)), source)
with raises(ValueError, match="Python3 mapping has no Excel counterpart"):  # a dict display: Python's alone
    Translators.between(PYTHON, EXCEL).forward(into(PATTERNS, PYTHON, term(E.mapping(("a", 1)))))